In [ ]:
from textwrap import fill

import cv2
import mediapipe as mp
import csv
import matplotlib.pyplot as plt
import time
import math
import parselmouth
from PIL.ImageOps import expand
from scipy.signal import find_peaks
import numpy as np
import pandas as pd
import seaborn as sns
from mediapipe.tasks.python import vision
from mediapipe.tasks import python
import os
import shutil

## Initialization

In [ ]:
tracks= ["NOSE","LEFT_SHOULDER","RIGHT_SHOULDER","LEFT_ELBOW","RIGHT_ELBOW","LEFT_WRIST","RIGHT_WRIST","LEFT_INDEX","RIGHT_INDEX"]
    
# Input videos
video_path = '../data/meteo4.mp4'
# Output csv file containing the (x,y,z) coordinates of the landmarks chosen among the ones listed in the file "landmark_names_list"
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "video_parts.csv")
keypoints = {
    0: 'NOSE',
    1: 'LEFT_EYE_INNER',
    2: 'LEFT_EYE',
    3: 'LEFT_EYE_OUTER',
    4: 'RIGHT_EYE_INNER',
    5: 'RIGHT_EYE',
    6: 'RIGHT_EYE_OUTER',
    7: 'LEFT_EAR',
    8: 'RIGHT_EAR',
    9: 'MOUTH_LEFT',
    10: 'MOUTH_RIGHT',
    11: 'LEFT_SHOULDER',
    12: 'RIGHT_SHOULDER',
    13: 'LEFT_ELBOW',
    14: 'RIGHT_ELBOW',
    15: 'LEFT_WRIST',
    16: 'RIGHT_WRIST',
    17: 'LEFT_PINKY',
    18: 'RIGHT_PINKY',
    19: 'LEFT_INDEX',
    20: 'RIGHT_INDEX',
    21: 'LEFT_THUMB',
    22: 'RIGHT_THUMB',
    23: 'LEFT_HIP',
    24: 'RIGHT_HIP',
    25: 'LEFT_KNEE',
    26: 'RIGHT_KNEE',
    27: 'LEFT_ANKLE',
    28: 'RIGHT_ANKLE',
    29: 'LEFT_HEEL',
    30: 'RIGHT_HEEL',
    31: 'LEFT_FOOT_INDEX',
    32: 'RIGHT_FOOT_INDEX'
}
keypoints_inverted = {v: k for k, v in keypoints.items()}

tracks_numbers = [keypoints_inverted[track] for track in tracks]

model_path = "../model/pose_landmarker_full.task"
base_options = python.BaseOptions(
    model_asset_path='../model/pose_landmarker_full.task', 
    # delegate=python.BaseOptions.Delegate.GPU
)

options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True)

landmarker = vision.PoseLandmarker.create_from_options(options)

## Extracting landmarks

In [ ]:
def extract_landmarks(results):
    row = []
    if results.pose_world_landmarks:
        landmarks = results.pose_world_landmarks
        for idx in tracks_numbers:
            lm = landmarks[0][idx]
            if lm.visibility > 0.5:
                row.extend([lm.x,lm.y,lm.z])
            else:
                row.extend([None,None,None])
    else:
        row.extend([None] * (len(tracks_numbers) * 3))
    return row

In [ ]:
def movement_extraction(video_path, landmarks, landmarker, outputs_destination, make_frames=False, keypoints = None):
    Path(outputs_destination).mkdir(parents=True, exist_ok=True)
    if keypoints is None:
        raise ValueError("Must set keypoints")
    keypoints_inverted = {v: k for k, v in keypoints.items()}
    tracks_numbers = [keypoints_inverted[landmark] for landmark in landmarks]
    outputs = []
    if isinstance(video_path, str):
        video_path = [video_path]
    for video in video_path:
        p = Path(video)
        outputs.append(Path(outputs_destination) / f"{p.stem}_parts.csv")
        cap = cv2.VideoCapture(video)
        frame_number = 0
        csv_data = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Create a MediaPipe Image from the NumPy RGB frame
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            # Process the frame with MediaPipe Pose
            result = landmarker.detect(mp_image)
            row = extract_landmarks(result)
            csv_data.append(row)

            # Draw only the tracked pose landmarks on the frame
            if result.pose_landmarks:
                for idx in tracks_numbers:
                    lm = result.pose_landmarks[0][idx]
                    h,w,_ = frame.shape
                    cx,cy = int(lm.x * w),int(lm.y * h)
                    cv2.circle(frame,(cx,cy),6,(0,255,0),-1)
            if make_frames:
                frame_path = Path(outputs_destination) / f"{p.stem}_frame_{frame_number}.png"
                cv2.imwrite(str(frame_path), frame)
                frame_number += 1
            # Exit if 'q' keypyt
        cap.release()
        print(f"Processed frames: {len(csv_data)}")
        output = outputs[-1]
        with open(output,'w',newline='') as f:
            writer = csv.writer(f)
            header = []
            for name in landmarks:
                header.extend([f"{name}_x",f"{name}_y",f"{name}_z"])
            writer.writerow(header)
            for row in csv_data:
                writer.writerow(row)
        video_base = p.stem
        movement_csv = Path(outputs_destination) / f"{video_base}_movement_all.csv"

        all_movement = {}
        all_accel = {}

        for lm_idx, lm_name in enumerate(landmarks):
            movement = [0.0]
            acceleration = [0.0]
            col = lm_idx * 3
            for i in range(1,len(csv_data)):
                prev = csv_data[i-1]
                curr = csv_data[i]
                if None not in (prev[col], curr[col]):
                    dx = curr[col] - prev[col]
                    dy = curr[col + 1] - prev[col + 1]
                    dz = curr[col + 2] - prev[col + 2]
                    dist = math.sqrt(dx*dx + dy*dy + dz*dz)
                else:
                    dist = 0.0
                movement.append(dist)
                accel = dist - movement[i - 1]
                acceleration.append(accel)
            all_movement[lm_name] = movement
            all_accel[lm_name]    = acceleration
        with open(movement_csv, 'w', newline='') as f:
            writer = csv.writer(f)
            movement_header = ['frame']
            for lm_name in landmarks:
                movement_header.extend([f'movement_{lm_name}', f'acceleration_{lm_name}'])
            writer.writerow(movement_header)
            for i in range(len(csv_data)):
                row = [i]
                for lm_name in landmarks:
                    row.extend([all_movement[lm_name][i], all_accel[lm_name][i]])
                writer.writerow(row)

## Plotting graphs for velocity and acceleration

In [ ]:
for lm in tracks:
    frames = []
    movement = []
    acceleration = []
    with open(f"meteo_movement_{lm}.csv", newline="") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            frames.append(int(row[0]))
            movement.append(float(row[1]))
            acceleration.append(float(row[2]))
    
    plt.figure(figsize=(15,8))
    plt.plot(frames, movement)
    plt.xlabel("Frame")
    plt.ylabel(f"Velocity of {lm}")
    plt.title(f"Evolution of velocity of {lm} throughout the video")
    plt.show()
    
    
    plt.figure(figsize=(15,8))
    plt.plot(frames, acceleration)
    plt.xlabel("Frame")
    plt.ylabel("Acceleration")
    plt.title(f"Evolution of acceleration of {lm} throughout the video")
    plt.show()

## Synching the audio with the gestures

In [ ]:
def nearest_event(t, events):
    return events[np.argmin(np.abs(events - t))]

snd = parselmouth.Sound("../data/meteoaudio.mp3")
    #Audio configs
pitch = snd.to_pitch(time_step=0.00333,pitch_floor=100.0,pitch_ceiling=500.0)
f0 = pitch.selected_array['frequency']
time_pitch = pitch.xs()
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
dt = 1/fps
np.savetxt(f"../outputs/audio_data.csv", np.column_stack((f0, time_pitch)), delimiter=",", header="f0,time_pitch", comments='')
for lm_name in tracks:
    movement_lm = pd.read_csv(f'meteo_movement_{lm_name}.csv')
    velocity_lm = movement_lm['movement']
    acceleration_lm = movement_lm['acceleration']
    time = [i * dt for i in range(len(movement_lm))]
    acceleration = np.array(acceleration_lm)
    velocity = np.array(velocity_lm)
    time = np.array(time)
    acc_threshold = np.percentile(acceleration,75)
    vel_threshold = np.percentile(velocity,75)
    peaks_acc, props_acc = find_peaks(acceleration, height=acc_threshold, distance=5)
    peak_acc_times = time[peaks_acc]
    peaks_vel, props_vel = find_peaks(velocity, height=vel_threshold, distance=5)
    peak_vel_times = time[peaks_vel]
    pitch_threshold = np.percentile(f0[f0 > 0], 75)
    peaks_pitch, _ = find_peaks(f0,height=pitch_threshold,distance=5)
    peak_pitch_times = time_pitch[peaks_pitch]
        
    D_acc = []
    D_vel = []
    for t_acc in peak_acc_times:
        t_pitch = nearest_event(t_acc, peak_pitch_times)
        D_acc.append(t_acc - t_pitch)
    D_acc = np.array(D_acc) * 1000
    for t_vel in peak_vel_times:
        t_pitch = nearest_event(t_vel, peak_pitch_times)
        D_vel.append(t_vel - t_pitch)
    D_vel = np.array(D_vel) * 1000
    fig, ax = plt.subplots(figsize=(8,4))
    sns.kdeplot(D_vel, ax=ax, color = 'orange', label=f"peak velocity ({lm_name})")
    sns.kdeplot(D_acc, ax=ax, color = 'green', label=f"peak acceleration ({lm_name})")
    ax.axvline(0, linestyle="--", color="blue", label="peak pitch")
    ax.set_xlabel("D (time in ms)")
    ax.set_ylabel("density")
    ax.set_title(f"Velocity & Acceleration vs Pitch — {lm_name}")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def histo(csv_path, f0_path, fps, threshold_percentile=75, peak_distance=5):
    audio_data = pd.read_csv(f0_path)
    f0 = audio_data['f0']
    time_pitch = audio_data['time_pitch']
    movement_lm = pd.read_csv(csv_path)
    acceleration = np.array(movement_lm['acceleration'])
    dt = 1 / fps
    time = np.array([i * dt for i in range(len(acceleration))])

    acc_threshold = np.percentile(acceleration, threshold_percentile)
    peaks_acc, _ = find_peaks(acceleration, height=acc_threshold, distance=peak_distance)
    peak_acc_times = time[peaks_acc]

    pitch_threshold = np.percentile(f0[f0 > 0], threshold_percentile)
    peaks_pitch, _ = find_peaks(f0, height=pitch_threshold, distance=peak_distance)
    peak_pitch_times = time_pitch[peaks_pitch]

    D = []
    for t_acc in peak_acc_times:
        t_pitch = peak_pitch_times[np.argmin(np.abs(peak_pitch_times - t_acc))]
        D.append(t_acc - t_pitch)
    D = np.array(D) * 1000

    fig, ax = plt.subplots(figsize=(8, 4))
    #sns.kdeplot(D, ax=ax, color='green', label='peak acceleration')
    sns.histplot(D, bins=10)
    ax.axvline(0, linestyle='--', color='blue', label='peak pitch')
    ax.set_xlabel('D (ms)')
    ax.set_ylabel('Density')
    ax.set_title(f'Acceleration peak vs Pitch peak — {csv_path}')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Smoothing the signals

In [ ]:
def smooth(x, window_len=11, window='hanning'):
    if x.ndim != 1:
        raise ValueError("smooth only accepts 1 dimension arrays")
    if x.size < window_len:
        raise ValueError("Input vector needs to be bigger than window size")
    if window_len < 3:
        return x
    if window not in ['flat', 'hanning', 'hamming', 'bartlett', 'blackman']:
        raise ValueError("Window must be one of 'flat', 'hanning', 'hamming', 'bartlett', 'blackman'")
    s = np.r_[x[window_len-1:0:-1], x, x[-2:-window_len-1:-1]]
    w = np.ones(window_len, 'd') if window == 'flat' else getattr(np, window)(window_len)
    y = np.convolve(w/w.sum(), s, mode='valid')
    trim = (len(y)- len(x))//2
    return y[trim:trim+len(x)]

In [ ]:
snd = parselmouth.Sound("../data/meteoaudio.mp3")
pitch = snd.to_pitch(time_step=0.00333, pitch_floor=100.0, pitch_ceiling=500.0)
f0 = pitch.selected_array['frequency']
time_pitch = pitch.xs()

for lm in tracks:
    frames = []
    accelerations = []
    with open(f"meteo_movement_{lm}.csv", newline="") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            frames.append(int(row[0]))
            accelerations.append(float(row[2]))

    frames = np.array(frames)
    acc_smooth = smooth(np.array(accelerations), window_len=11, window='hanning')
    threshold = 0.75 * acc_smooth.max()
    mask = acc_smooth >= threshold
    frames_filtered = np.array(frames)[mask]
    frames_discarded = np.array(frames)[~mask]

    plt.figure(figsize=(15, 8))
    plt.plot(frames, acc_smooth, label="Smoothed acceleration", color="steelblue")
    plt.xlabel("Frame")
    plt.ylabel("Acceleration")
    plt.title(f"Smoothed acceleration")
    plt.legend()
    plt.tight_layout()
    plt.show()
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    movement = pd.read_csv('meteo_movement_RIGHT_INDEX.csv')
    dt = 1 / (fps+1e-5)
    time = np.array([i * dt for i in range(len(movement))])

    acc_raw = np.array(movement['acceleration'])
    acc_smoothed = smooth(acc_raw, window_len=11, window='hanning')

    pitch_threshold = np.percentile(f0[f0 > 0], 75)
    peaks_pitch, _ = find_peaks(f0, height=pitch_threshold, distance=5)
    peak_pitch_times = time_pitch[peaks_pitch]

    #original acceleration
    acc_threshold_raw = np.percentile(acc_raw, 75)
    peaks_raw, _ = find_peaks(acc_raw, height=acc_threshold_raw, distance=5)
    peak_acc_times_raw = time[peaks_raw]

    D_raw = []
    for t_acc in peak_acc_times_raw:
        t_pitch = nearest_event(t_acc, peak_pitch_times)
        D_raw.append(t_acc - t_pitch)
    D_raw = np.array(D_raw) * 1000

    #Smooth acceleration
    acc_threshold_smooth = np.percentile(acc_smoothed, 75)
    peaks_smooth, _ = find_peaks(acc_smoothed, height=acc_threshold_smooth, distance=5)
    peak_acc_times_smooth = time[peaks_smooth]

    D_smooth = []
    for t_acc in peak_acc_times_smooth:
        t_pitch = nearest_event(t_acc, peak_pitch_times)
        D_smooth.append(t_acc - t_pitch)
    D_smooth = np.array(D_smooth) * 1000

    plt.figure(figsize=(12, 6))
    sns.kdeplot(D_raw, label=f"Raw acceleration ({lm})", linestyle="--", color="tomato")
    sns.kdeplot(D_smooth, label=f"Smoothed acceleration ({lm})", linestyle="-", color="steelblue")
    plt.axvline(0, linestyle="--", color="blue", linewidth=1.2, label="Peak pitch (ref)")
    plt.xlabel("D (time in ms)")
    plt.ylabel("Density")
    plt.legend()
    plt.title(f"Acceleration peak vs Pitch peak ({lm}): Raw vs Smoothed")
    plt.tight_layout()
    plt.show()

# Interface

In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog
from PIL import Image, ImageTk, ImageOps
import vlc
from datetime import date

#TODO give a type to every variable that isn't defined

DATA_PATH = "./../data/participants/"
THIS_YEAR = date.today().year

BACKGROUND = "darkgray"


def test():
    print("===== TEST =====")


def put_image(image_path, frame, label):
    """
    Put an image in the given label, resized to fit the given frame.

    Args:
        image_path: Path to the image
        frame: Frame whose size is used for resizing
        label: Label where the image will be displayed
    """
    if os.path.isfile(image_path):
        frame.update_idletasks()
        image_import = Image.open(image_path)
        image_resized = ImageOps.contain(
            image_import,
            (frame.winfo_width(), frame.winfo_height())
        )
        image = ImageTk.PhotoImage(image_resized)
        label.config(
            image=image
        )
        label.image = image #TODO check the usefulness of that line
    else:
        label.config(text="ERROR : Image not found")


class Application(tk.Frame):
    # ====================
    # ===== __init__ =====
    # ====================
    def __init__(self, master=None):
        super().__init__(master)
        self.master = master
        self.pack(fill="both", expand=True)

        # ============================
        # ===== GLOBAL VARIABLES =====
        self.popup = None

        self.PARTICIPANT_NAME = None
        self.PARTICIPANT_AGE = None
        self.DATA_PATH_PARTICIPANT = None
        self.DATA_PATH_ID = None
        self.VIDEO_YEAR = None

        #TODO ranger dans l'ordre d'apparition
        self.error_label = None
        self.name_entry = None
        self.peak_distance = None
        self.peak_percentage = None
        self.f0_frequency = None
        self.video_year_label = None
        self.participant_age_label = None
        self.participant_name_label = None
        self.body_part_label = None
        self.body_part_frame = None
        self.video_frame = None
        self.vlc_player = None
        self.body_part = None
        self.image_resized = None
        self.image_import = None
        self.body_part_selection = None
        self.participant_id_drop_down_list = None
        self.vlc_instance = None
        self.video_year_entry = None
        self.age_entry = None
        self.video_path_entry = None
        self.history_drop_down_list = None

        self.third_of_width = 500
        self.label_width = 16

        self.is_video_playing = True
        self.display_help_at_start = True
        # ============================

        self.build_header()
        self.build_body()

    # ===================
    # ===== BACK =====
    # ==============================================================================================================================================================================================================
    # ==============================================================================================================================================================================================================

    def form_validation(self):
        """
        Gets the values from the form to create the new participant's structure and set values with them
        """
        print("FORM VALIDATION :") #TODO FORM VALIDATION
        if self.name_entry.get() and (self.video_path_entry.get() and os.path.exists(self.video_path_entry.get())) and (self.video_year_entry.get() and len(self.video_year_entry.get()) == 4):
            print("    ALL VALUES ENTERED AND CORRECT")
            print("    (create participant tree)")
            ...
            print("    (set values)")
            ...
            self.popup.destroy()
        else:
            if not os.path.exists(self.video_path_entry.get()):
                self.error_label.config(
                    text="ERROR : Video not found"
                )
            elif(not len(self.video_year_entry.get()) == 4):
                self.error_label.config(
                    text="ERROR : Wrong year format"
                )
            else:
                self.error_label.config(
                    text="ERROR : All field are not filled"
                )

            self.error_label.pack(
                side="top",
                fill="y",
                expand=True,
                anchor="center",
            )
            #TODO make a special label when all the form's fields are not filled


    def create_participant_tree(self, name="name", age=8, video_path="./../data/meteo4.mp4", video_year=2026):
        """
        Initialize the data for a new participant:
        - Create the participant's folders
        - Write the data to a CSV file
        - Copy the video given to the participant's folder as "video.mp4"
        """
        participant = name + "-" + str(video_year)
        self.DATA_PATH_PARTICIPANT = DATA_PATH + participant + "/"
        if not os.path.exists(self.DATA_PATH_PARTICIPANT):
            os.mkdir(self.DATA_PATH_PARTICIPANT)

        id = len([
            e for e in os.listdir(self.DATA_PATH_PARTICIPANT)
            if os.path.isdir(os.path.join(self.DATA_PATH_PARTICIPANT, e))
            #HACK I've found this bit of code to only get dirs #TODO try to find a better way to only get dirs
        ])+1

        self.DATA_PATH_ID = self.DATA_PATH_PARTICIPANT + str(id) + "/"
        os.mkdir(self.DATA_PATH_ID)

        # ===== WRITING THE CSV =====
        with open(self.DATA_PATH_PARTICIPANT + "data.csv", "w", encoding="utf-8") as file:  #TODO check if we can use self.DATA_PATH_PARTICIPANT here
            columns = ["name", "age", "video year"]
            writer = csv.DictWriter(file, fieldnames=columns, delimiter=";")
            writer.writeheader()

            writer.writerow({
                "name": name,
                "age": age,
                "video year": video_year
            })

        shutil.copy(video_path, self.DATA_PATH_ID + "video.mp4")

    # ==========================
    # ===== VIDEO CONTROLS =====
    # ==========================
    def play(self):
        self.vlc_player.play()

    def pause(self):
        self.vlc_player.pause()

    def load_video(self, path):
        media = self.vlc_instance.media_new(path)
        self.vlc_player.set_media(media)

        self.master.update()
        self.vlc_player.set_xwindow(self.video_frame.winfo_id())
        self.play()
        self.after(1,
                   self.pause)  #HACK to have the video showing we need to play it, so right after it start we pause it.

    def play_pause_button_command(self):
        if self.is_video_playing:
            self.pause()
        else:
            self.play()

    # ==================
    # ===== POPUPS =====
    # ==============================================================================================================================================================================================================
    # ==============================================================================================================================================================================================================
    # ===== NAME ENTRY =====
    def build_popup_name_entry(self, parent, width):
        frame = tk.Frame(
            parent
        )
        frame.pack(side="top", fill="x")
        tk.Label(
            frame,
            text="name :",
            anchor="e",
            width=width
        ).pack(
            side="left",
            padx=5,
            pady=5
        )
        self.name_entry = tk.Entry(
            frame,
            background="white"
        )
        self.name_entry.pack(
            side="left",
            padx=5,
            pady=5
        )

    # ======================
    # ===== AGE ENTRY =====
    def build_popup_age_entry(self, parent, width):
        frame = tk.Frame(
            parent
        )
        # frame.pack(
        #     side="top",
        #     fill="x"
        # )
        tk.Label(
            frame,
            text="age :",
            anchor="e",
            width=width
        ).pack(
            side="left",
            padx=5,
            pady=5
        )

        self.age_entry = tk.Entry(
            frame,
            background="white",
            width=3,
            validate="key",
            validatecommand=(
                self.register(
                    lambda value: (value.isdigit() or value == "") and len(value) <= 2
                ),
                "%P"
            )
        )
        self.age_entry.pack(
            side="left",
            padx=5,
            pady=5
        )

    # =======================
    # ===== VIDEO ENTRY =====
    def browse(self):
        path = filedialog.askopenfilename(
            parent=self.popup,
            filetypes=[("Video", "*.mp4")]
        )
        if path:
            if os.path.exists(path):
                if self.video_path_entry.get():
                    self.video_path_entry.delete(0, tk.END)
                    self.video_path_entry.insert(0, path)
                else:
                    self.video_path_entry.insert(0, path)

    def build_popup_video_selector_entry(self, parent, width):
        frame = tk.Frame(
            parent
        )
        frame.pack(
            side="top",
            fill="x"
        )
        tk.Label(
            frame,
            text="video :",
            anchor="e",
            width=width
        ).pack(
            side="left",
            padx=5,
            pady=5
        )
        self.video_path_entry = tk.Entry(
            frame,
            bg="white",
            width=50
        )
        self.video_path_entry.pack(
            side="left",
            padx=5,
            pady=5
        )
        video_selector = tk.Button(
            frame,
            bg=BACKGROUND,
            text="Browse...",
            command=self.browse
        )
        video_selector.pack(
            side="left",
            padx=5,
            pady=5
        )

    # ============================
    # ===== VIDEO YEAR ENTRY =====
    def build_popup_video_year_entry(self, parent, width):
        frame = tk.Frame(
            parent
        )
        frame.pack(
            side="top",
            fill="x"
        )
        tk.Label(
            frame,
            text="video year :",
            anchor="e",
            width=width
        ).pack(
            side="left",
            padx=5,
            pady=5
        )

        self.video_year_entry = tk.Entry(
            frame,
            background="white",
            width=4,
            validate="key",
            validatecommand=(
                self.register(
                    lambda value: (value.isdigit() or value == "") and len(value) <= 4
                ),
                "%P"
            )
        )
        self.video_year_entry.pack(
            side="left",
            padx=5,
            pady=5
        )

    def build_popup_footer(self, parent):
        frame = tk.Frame(
            parent,
            bg=BACKGROUND
        )
        frame.pack(
            side="bottom",
            fill="x",
        )

        self.error_label = tk.Label(
            frame,
            borderwidth=2,
            relief="raised",
            bg="red",
            padx=5,
        )

        tk.Button(
            frame,
            fg="red",
            text="Cancel",
            command=self.popup.destroy
        ).pack(
            side="right"
        )
        tk.Button(
            frame,
            fg="green",
            text="Confirm",
            command=self.form_validation
        ).pack(
            side="right",
        )

    # ======================

    def build_popup_info(self):
        self.popup = tk.Toplevel(root)
        self.popup.title("Infos")
        self.popup.geometry("+700+400")
        self.popup.lift()
        self.popup.attributes("-topmost", True)
        self.popup.attributes("-topmost", False)

        tk.Label(
            self.popup,
            text="info",
        ).pack()
        tk.Button(
            self.popup,
            fg="red",
            bg=BACKGROUND,
            text="Back",
            command=self.popup.destroy
        ).pack()

    def build_popup_form(self):
        self.popup = tk.Toplevel(root)
        self.popup.title("New Participant")
        self.popup.geometry("+700+400")
        self.popup.lift()
        self.popup.attributes("-topmost", True)
        self.popup.attributes("-topmost", False)

        labels_width = 10

        self.build_popup_name_entry(self.popup, labels_width)
        self.build_popup_age_entry(self.popup, labels_width)
        self.build_popup_video_selector_entry(self.popup, labels_width)
        self.build_popup_video_year_entry(self.popup, labels_width)

        self.build_popup_footer(self.popup)





    # =====================
    # ===== MAIN PART =====
    # ==============================================================================================================================================================================================================
    # ==============================================================================================================================================================================================================
    # ===== HEADER =====
    # ==================

    # ===== FORM & IMPORT =====
    def build_header_from_and_import(self, parent):
        frame = tk.Frame(
            parent,
            width=self.third_of_width,
            # bg="#FF0000",
            height=40
        )
        frame.pack(
            side="left",
        )
        frame.pack_propagate(False)
        tk.Button(
            frame,
            bg=BACKGROUND,
            text="New Participant",
            command=self.build_popup_form
        ).pack(
            side="left"
        )

        # ===== PARTICIPANTS HISTORY DROP-DOWN LIST =====
        history = os.listdir("./../data/participants/")
        self.history_drop_down_list = ttk.Combobox(
            frame,
            values=history
        )
        self.history_drop_down_list.set("--participants history--")
        self.history_drop_down_list.pack(
            side="left"
        )
        # self.history_drop_down_list.bind("<<ComboboxSelected>>", self.history_bind_function)
        # ============================================

        # ===== PARTICIPANT ID DROP-DOWN LIST ======
        self.participant_id_drop_down_list = ttk.Combobox(
            frame,
            #width=5,
        )
        self.participant_id_drop_down_list.set("--id--")
        self.participant_id_drop_down_list.pack(
            side="left"
        )
        # ===========================

    # ===== QUIT BUTTON =====
    def build_header_quit_button(self, parent):
        frame = tk.Frame(
            parent,
            # bg="#0000FF"
        )
        frame.pack(
            side="left",
            fill="both",
            expand=True
        )

        tk.Button(
            frame,
            fg="red",
            bg=BACKGROUND,
            text="QUIT",
            command=self.master.destroy
        ).pack(
            side="right"
        )

        tk.Button(
            frame,
            fg="blue",
            bg=BACKGROUND,
            text="Reload",
            #command=self.update_from_constants
        ).pack(
            side="right"
        )

    def build_header(self):
        header = tk.Frame(
            self,
            height=40
        )
        header.pack(side="top", fill="x")
        header.pack_propagate = False

        self.build_header_from_and_import(header)
        self.build_header_quit_button(header)

    # ==================

    # ================
    # ===== BODY =====
    def build_body(self):
        # =========================
        # ===== FRAME DU BODY =====
        body = tk.Frame(
            self
        )
        body.pack(fill="both", expand=True)
        # =========================

        self.build_body_video(body)

        #HACK instead of having to deal with puting the combobox in grid, I have made 2 more Frames to mimic a grid system
        #TODO put the whole thing in a grid system

        # =============================================
        # ===== LISTE DÉROULANTE PARTIES DU CORPS =====
        self.body_part_selection = ttk.Combobox(
            body,
            values=tracks
        )
        self.body_part_selection.set("--body parts--")
        self.body_part_selection.pack(
            side="top",
            fill="x"
        )
        self.body_part_selection.bind("<<ComboboxSelected>>", self.body_part_selectioner)
        # =============================================
        self.build_body_graphs(body)
        self.build_body_histogram_and_option(body)

    def body_part_selectioner(self, event):
        path = DATA_PATH + "/" + self.body_part_selection.get().lower() + ".png"
        if os.path.isfile(path):
            self.image_import = Image.open(path)
            self.image_resized = ImageOps.contain(
                self.image_import,
                (self.body_part_frame.winfo_width(), self.body_part_frame.winfo_height())
            )
            self.body_part = ImageTk.PhotoImage(self.image_resized)
            self.body_part_label.config(image=self.body_part)
        else:
            self.body_part_label.config(text="ERROR : Image not found")

    # =================
    # ===== VIDEO =====
    def build_body_video(self, parent):
        frame = tk.Frame(
            parent,
            bg="#AF0000",
            width=self.third_of_width,
            padx=10,
            pady=10
        )
        frame.pack(
            side="left",
            fill="y",
        )
        frame.pack_propagate(False)
        self.build_video_player(frame)
        self.build_video_controls(frame)

    def build_video_player(self, parent):
        self.vlc_instance = vlc.Instance()
        self.vlc_player = self.vlc_instance.media_player_new()

        self.video_frame = tk.Frame(
            parent
        )
        self.video_frame.pack(
            side="top",
            fill="both",
            expand=True
        )
        self.video_frame.pack_propagate(False)

    def build_video_controls(self, parent):
        frame = tk.Frame(
            parent,
        )
        frame.pack(
            side="top",
            fill="x"
        )

        play_pause_button = tk.Button(
            frame,
            bg=BACKGROUND,
            text="PlayPause",
            command=self.play_pause_button_command
        )
        play_pause_button.pack(
            side="left"
        )

    # =================

    # ==================================
    # ===== ACCELERATION CURVES =====
    def build_body_graphs(self, parent):
        frame = tk.Frame(
            parent,
            bg="#00AF00",
            width=self.third_of_width,
            padx=10,
            pady=10
        )
        frame.pack(side="left", fill="y")
        frame.pack_propagate(False)

        self.build_body_parts_curves(frame)
        self.build_graphes_f0(frame)

    # ===== MOVE =====
    def build_body_parts_curves(self, parent):
        self.body_part_frame = tk.Frame(
            parent,
        )
        self.body_part_frame.pack(
            side="top",
            fill="both",
            expand=True
        )
        self.body_part_frame.pack_propagate(False)

        self.body_part_label = tk.Label(
            self.body_part_frame
        )
        self.body_part_label.pack(
            fill="both"
        )

    # ===== f0 =====
    def build_graphes_f0(self, parent):
        frame = tk.Frame(
            parent,
        )
        frame.pack(side="top", fill="both", expand=True)
        frame.pack_propagate(False)
        f0_label = tk.Label(
            frame
        )
        f0_label.pack(
            fill="both"
        )

    # ===================

    # ===============================
    # ===== HISTOGRAM & OPTIONS =====
    def build_body_histogram_and_option(self, parent):
        frame = tk.Frame(
            parent,
            bg="#0000AF",
            # width=self.third_of_width,
            padx=10,
            pady=10
        )
        frame.pack(
            side="left",
            fill="both",
            expand=True
        )
        frame.pack_propagate(False)

        self.build_histogram(frame)
        self.build_options(frame)

    # ===== HISTOGRAM =====
    def build_histogram(self, parent):
        frame = tk.Frame(
            parent,
        )
        frame.pack(
            side="top",
            fill="both",
            expand=True
        )
        frame.pack_propagate(False)

    # ===================
    # ===== OPTIONS =====
    def build_options(self, parent):
        frame = tk.Frame(
            parent,
            borderwidth=3,
            relief="groove"
        )
        frame.pack(side="top", fill="both", expand=True)
        frame.pack_propagate(False)

        self.option_participant_name(frame)
        # self.option_participant_age(frame)
        self.option_video_year(frame)

        # ======================
        # ===== SEPARATION =====
        ttk.Separator(
            frame,
            orient="horizontal"
        ).pack(fill="x")
        # ======================

        self.option_f0_frequency(frame)
        self.option_peak_percentage(frame)
        self.option_peak_distance(frame)

    # ===== PARTICIPANT NAME =====
    def option_participant_name(self, parent):
        frame = tk.Frame(
            parent
        )
        frame.pack(
            side="top",
            fill="x"
        )

        tk.Label(
            frame,
            text="participant name : ",
            width=self.label_width,
            anchor="e"
        ).pack(
            side="left"
        )

        self.participant_name_label = tk.Label(
            frame
        )
        self.participant_name_label.pack(
            side="left"
        )
        if self.PARTICIPANT_NAME:
            self.participant_name_label.config(
                text=self.PARTICIPANT_NAME
            )
        else:
            self.participant_name_label.config(
                text=" - "
            )

    # ===== PARTICIPANT AGE =====
    def option_participant_age(self, parent):
        frame = tk.Frame(
            parent
        )
        frame.pack(
            side="top",
            fill="x"
        )

        tk.Label(
            frame,
            text="participant age : ",
            width=self.label_width,
            anchor="e"
        ).pack(
            side="left"
        )

        self.participant_age_label = tk.Label(
            frame
        )
        self.participant_age_label.pack(
            side="left"
        )
        if self.PARTICIPANT_AGE:
            self.participant_age_label.config(
                text=self.PARTICIPANT_AGE
            )
        else:
            self.participant_age_label.config(
                text=" - "
            )

    # ===== VIDEO YEAR =====
    def option_video_year(self, parent):
        frame = tk.Frame(
            parent
        )
        frame.pack(
            side="top",
            fill="x"
        )

        tk.Label(
            frame,
            text="video year : ",
            width=self.label_width,
            anchor="e"
        ).pack(
            side="left"
        )

        self.video_year_label = tk.Label(
            frame,
            text=str(self.VIDEO_YEAR)
        )
        self.video_year_label.pack(
            side="left"
        )

    # ===== F0 FREQUENCY SELECTION =====
    def option_f0_frequency(self, parent):
        frame = tk.Frame(
            parent
        )
        frame.pack(
            side="top",
            fill="x"
        )

        tk.Label(
            frame,
            text="f0's frequency : ",
            width=self.label_width,
            anchor="e"
        ).pack(
            side="left"
        )

        self.f0_frequency = tk.Entry(
            frame,
            bg="white",
            width=3,
            validate="all",
            validatecommand=(
                self.register(
                    lambda value: (value.isdigit() or value == "") and len(value) <= 3
                ),
                "%P"
            )
        )
        self.f0_frequency.pack(
            side="left"
        )

        tk.Label(
            frame,
            text="Hz"
        ).pack(
            side="left"
        )

    # ===== PEAK PERCENTAGE SELECTION =====
    def option_peak_percentage(self, parent):
        frame = tk.Frame(
            parent
        )
        frame.pack(
            side="top",
            fill="x"
        )

        tk.Label(
            frame,
            text="Peak percentage : ",
            width=self.label_width,
            anchor="e"
        ).pack(
            side="left"
        )

        self.peak_percentage = tk.Entry(
            frame,
            bg="white",
            width=3,
            validate="all",
            validatecommand=(
                self.register(
                    lambda value: (value.isdigit() or value == "") and len(value) <= 3
                ),
                "%S"
            )
        )
        self.peak_percentage.pack(
            side="left"
        )

        tk.Label(
            frame,
            text="%"
        ).pack(
            side="left"
        )

    # ===== PEAK DISTANCE =====
    def option_peak_distance(self, parent):
        frame = tk.Frame(
            parent
        )
        frame.pack(
            side='top',
            fill="x"
        )
        tk.Label(
            frame,
            text="Peak distance : ",
            width=self.label_width,
            anchor="e"
        ).pack(
            side="left"
        )

        self.peak_distance = tk.Entry(
            frame,
            bg="white",
            width=3,
            validate="all",
            validatecommand=(
                self.register(
                    lambda value: value.isdigit() or value == ""
                ),
                "%P"
            )
        )
        self.peak_distance.pack(
            side="left"
        )

    # ===================
    # ===============================
    # ================


root = tk.Tk()
root.title("MOVEMENT & SPEACH ANALYSER")
root.geometry("1500x800+200+150")
# root.maxsize(1500, 800)
# root.minsize(1500, 800)
app = Application(master=root)
app.mainloop()